In [3]:
import csv
import re
import numpy as np
import pandas as pd

csv_path = "8891_motorcycles.csv"
EXPECTED_COLS = [
    "category", "title", "brand", "model", "price_twd",
    "year", "gas_displacement", "region", "raw_price", "url",
]

# 1. Line-by-line resilient CSV parser (prevents unclosed '"' from merging rows
#    AND preserves commas inside 'title')
parsed_rows = []
skipped_lines = []

with open(csv_path, "r", encoding="utf-8-sig", errors="replace", newline="") as f:
    raw_lines = [line.rstrip("\r\n") for line in f if line.strip()]

for line_idx, line in enumerate(raw_lines[1:], start=2):  # skip header line 1
    try:
        fields = next(csv.reader([line], quotechar='"', quoting=csv.QUOTE_MINIMAL))
    except csv.Error:
        fields = line.split(",")

    if len(fields) == 10:
        parsed_rows.append(fields)
    elif len(fields) > 10:
        # Anchor column 0 (category) and the last 8 columns (brand..url);
        # rejoin any extra comma-separated pieces back into 'title'
        reconstructed = [fields[0], ",".join(fields[1:-8]), *fields[-8:]]
        parsed_rows.append(reconstructed)
    else:
        skipped_lines.append((line_idx, len(fields), line[:120]))

df = pd.DataFrame(parsed_rows, columns=EXPECTED_COLS)
for col in df.columns:
    df[col] = df[col].astype(str).str.strip().str.strip('"')

# 2. Print the exact 4-stage funnel diagnostic
print("=== ROW FUNNEL DIAGNOSTIC ===")
print(f"Stage 1 - Physical data lines in CSV:        {len(raw_lines) - 1}")
print(f"Stage 2 - Successfully parsed rows:          {len(df)} (skipped malformed: {len(skipped_lines)})")
print(f"Stage 3a - Unique URLs:                      {df['url'].nunique()}")
print(f"Stage 3b - Unique (title, price, year, reg): {len(df.drop_duplicates(subset=['title', 'price_twd', 'year', 'region']))}")

# 3. Deduplicate & clean
clean_df = df.drop_duplicates(
    subset=["title", "price_twd", "year", "region"]
).reset_index(drop=True)

CURRENT_YEAR = 2026
PLACEHOLDER_PRICES = {1111, 1234, 8888, 9999, 12345, 88888, 99999, 123456, 888888, 999999}

def clean_price(val) -> float:
    if pd.isna(val):
        return np.nan
    s = str(val).replace(",", "").replace(" ", "").strip()
    m = re.search(r"(\d+(?:\.\d+)?)\s*(萬)?", s)
    if not m:
        return np.nan
    num = float(m.group(1)) * (10_000 if m.group(2) == "萬" else 1)
    price = int(round(num))
    return float(price) if (5_000 <= price <= 3_500_000 and price not in PLACEHOLDER_PRICES) else np.nan

def clean_year(val) -> float:
    if pd.isna(val):
        return np.nan
    m = re.search(r"(\d{2,4})", str(val))
    if not m:
        return np.nan
    yr = int(m.group(1))
    if 70 <= yr <= 116:
        yr += 1911
    return float(yr) if 1980 <= yr <= CURRENT_YEAR + 1 else np.nan

def extract_cc(cc_val, title: str, brand: str) -> float:
    if brand == "GOGORO":
        return 0.0
    if pd.notna(cc_val):
        s_cc = str(cc_val).strip()
        m_liter = re.search(r"(\d+(?:\.\d+)?)\s*l\b", s_cc, flags=re.IGNORECASE)
        if m_liter:
            cc_num = round(float(m_liter.group(1)) * 1000)
            if 49 <= cc_num <= 2500:
                return float(cc_num)
        m_num = re.search(r"(\d{2,4})", s_cc)
        if m_num and 49 <= int(m_num.group(1)) <= 2500:
            return float(m_num.group(1))
    if pd.notna(title):
        m_title = re.search(r"\b(\d{2,4})\s*(?:cc|c\.c\.)\b", str(title), flags=re.IGNORECASE)
        if m_title and 49 <= int(m_title.group(1)) <= 2500:
            return float(m_title.group(1))
    return np.nan

def extract_mileage_from_title(title: str) -> float:
    if pd.isna(title):
        return np.nan
    s = str(title).replace(",", "").strip()
    m = re.search(r"(?:實跑|里程|僅跑|只跑)\s*(\d+(?:\.\d+)?)\s*(萬|千)?\s*(?:公里|km)?", s, flags=re.IGNORECASE)
    if not m:
        return np.nan
    mult = 10_000 if m.group(2) == "萬" else (1_000 if m.group(2) == "千" else 1)
    km = int(round(float(m.group(1)) * mult))
    return float(km) if 0 <= km <= 300_000 else np.nan

BRAND_MAP = {
    "SYM": ["sym", "三陽"], "KYMCO": ["kymco", "光陽"], "YAMAHA": ["yamaha", "山葉"],
    "GOGORO": ["gogoro", "睿能"], "PGO": ["pgo", "摩特動力", "比雅久"], "AEON": ["aeon", "宏佳騰"],
    "SUZUKI": ["suzuki", "台鈴", "鈴木"], "HONDA": ["honda", "本田"], "KAWASAKI": ["kawasaki", "川崎"],
    "VESPA": ["vespa", "偉士牌"], "DUCATI": ["ducati", "杜卡迪"], "BMW": ["bmw"],
}

def standardize_brand(text: str) -> str:
    if pd.isna(text):
        return "OTHER"
    s = str(text).lower()
    for canonical, aliases in BRAND_MAP.items():
        if any(alias in s for alias in aliases):
            return canonical
    return "OTHER"

clean_df["price_ntd"] = clean_df["price_twd"].apply(clean_price).fillna(clean_df["raw_price"].apply(clean_price))
clean_df["model_year"] = clean_df["year"].apply(clean_year)
clean_df["vehicle_age"] = CURRENT_YEAR - clean_df["model_year"]
clean_df["mileage_km"] = clean_df["title"].apply(extract_mileage_from_title)
clean_df.loc[(clean_df["mileage_km"] == 0) & (clean_df["vehicle_age"] >= 1), "mileage_km"] = np.nan

clean_df["brand_clean"] = (clean_df["brand"].fillna("") + " " + clean_df["title"].fillna("")).apply(standardize_brand)
clean_df["model_clean"] = clean_df["model"].fillna("OTHER").astype(str).str.strip().str.upper().replace({"": "OTHER", "NAN": "OTHER"})
clean_df["cc_clean"] = [extract_cc(c, t, b) for c, t, b in zip(clean_df["gas_displacement"], clean_df["title"], clean_df["brand_clean"])]
clean_df["region_clean"] = clean_df["region"].astype(str).str.strip().replace({"nan": np.nan, "": np.nan})

clean_df = clean_df.dropna(subset=["price_ntd"]).reset_index(drop=True)
print(f"Stage 4 - Final valid cleaned rows:          {len(clean_df)}")

# Export updated Parquet and CSV
clean_df.to_parquet("8891_motorcycles_clean.parquet", index=False)
clean_df.to_csv("8891_motorcycles_clean.csv", index=False, encoding="utf-8-sig")

=== ROW FUNNEL DIAGNOSTIC ===
Stage 1 - Physical data lines in CSV:        1500
Stage 2 - Successfully parsed rows:          1500 (skipped malformed: 0)
Stage 3a - Unique URLs:                      30
Stage 3b - Unique (title, price, year, reg): 30
Stage 4 - Final valid cleaned rows:          30
